# 🔬 Napari image inspection

A clean, step-by-step workflow for opening an OME-TIFF, reading its metadata, optionally setting custom display ranges, and viewing channels in napari.

**Workflow:** ⚙️ setup → 📁 input → 📦 load → 📐 metadata → 🎚️ contrast → 🖥️ viewer


## ⚙️ Prepare the environment

Set the GUI-related environment variables required by napari in this session.

In [1]:
import os
os.environ["DISPLAY"] = ":1"
os.environ["XDG_RUNTIME_DIR"] = "/tmp/runtime-jovyan"
os.environ["QT_QPA_PLATFORM"] = "xcb"

In [ ]:
import napari
from bioio import BioImage

# Add [minimum, maximum] values in channel order after running the contrast cell.
CUSTOM_CONTRAST_LIMITS = []


## 📁 Choose the input image

Point the workflow to the OME-TIFF that should be inspected. Update `file_path` when working with another dataset.

In [ ]:
from pathlib import Path

# 📁 Choose the input image
file_path = Path("your_image_file.ome.tif")
input_path = file_path.expanduser().resolve()
print(f"Input image: {input_path}")

## 📦 Load the image data

Read the first timepoint in `CZYX` order so the channel axis is explicit for later display.

In [ ]:
img = BioImage(input_path)
data = img.get_image_data("CZYX", T=0)

print(f"Loaded data with shape CZYX = {data.shape}")

## 📐 Read physical scale and channel metadata

Extract voxel spacing, channel names, and OME-defined colors so napari can display the image in physical coordinates with meaningful labels.

In [ ]:
scale = img.physical_pixel_sizes
spatial_scale = [scale.Z, scale.Y, scale.X]

ome = img.ome_metadata
channels_meta = ome.images[0].pixels.channels
channel_names = [channel.name or f"Channel {index}" for index, channel in enumerate(channels_meta)]

colormaps = []
for channel in channels_meta:
    if channel.color is not None:
        red, green, blue = channel.color.as_rgb_tuple()[:3]
        colormaps.append([[0, 0, 0], [red / 255, green / 255, blue / 255]])
    else:
        colormaps.append("gray")

print(f"Channels: {', '.join(channel_names)}")
print(f"Spatial scale (Z, Y, X): {spatial_scale}")

## 🎚️ Set optional display contrast

Run this cell once to print a copy-pasteable `CUSTOM_CONTRAST_LIMITS` list. Replace the `None` entries with `[minimum, maximum]` values in channel order, then paste the list into the setup cell. Leave entries as `None` to keep napari's automatic contrast limits.


In [ ]:
if len(CUSTOM_CONTRAST_LIMITS) > len(channel_names):
    raise ValueError(
        f"CUSTOM_CONTRAST_LIMITS has {len(CUSTOM_CONTRAST_LIMITS)} entries, "
        f"but the image has {len(channel_names)} channels."
    )

contrast_limits = list(CUSTOM_CONTRAST_LIMITS)
contrast_limits.extend([None] * (len(channel_names) - len(contrast_limits)))

print("CUSTOM_CONTRAST_LIMITS = [")
for index, (name, limits) in enumerate(zip(channel_names, contrast_limits)):
    value = repr(limits) if limits is not None else "None"
    print(f"    {value},  # Channel {index}: {name}")
print("]")


## 🖥️ Display image in napari

Create one viewer with additive channel blending, OME colors, physical scaling, and any custom contrast limits.


In [ ]:
viewer = napari.Viewer()

image_kwargs = {
    "channel_axis": 0,
    "scale": spatial_scale,
    "name": channel_names,
    "colormap": colormaps,
    "blending": "additive",
}
layers = viewer.add_image(data, **image_kwargs)
for index, limits in enumerate(contrast_limits):
    if limits is not None:
        layers[index].contrast_limits = limits
